# 02 — Exploratory Analysis

**Project:** Repeat Purchase Recovery — Olist E-commerce (Brazil)  
**Objective:** Establish the baseline picture of customer behaviour — how many customers return, when they return, and what the revenue gap looks like between repeat and one-time buyers.  
**Data source:** `data/olist.db` — validated layers through Step 5 (materialized mart tables)  
**Observation window:** September 2016 – August 2018

---

### What this notebook answers

| # | Question |
|---|----------|
| 1 | How many customers and valid orders are in the dataset? |
| 2 | What fraction of customers are repeat buyers vs one-time buyers? |
| 3 | What does the distribution of orders per customer look like? |
| 4 | How has first-order acquisition trended monthly? |
| 5 | How does the second-order volume track against first orders? |
| 6 | What are the headline repeat purchase rates (30 / 60 / 90 day windows)? |

## Setup

Connect to the SQLite database and configure chart styling.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Database connection ───────────────────────────────────────────────────────
DB_PATH = "../data/olist.db"
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

# ── Chart style ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.facecolor": "white",
})

# Colour palette
C_BLUE   = "#2563EB"
C_ORANGE = "#EA580C"
C_GREY   = "#94A3B8"
C_GREEN  = "#16A34A"

print("Connected to:", DB_PATH)

---

## 1. Dataset Overview

Start with a simple count of customers, orders, and the observation window. This confirms the scope of the analysis before going deeper.

In [ ]:
overview_query = """
SELECT
    COUNT(DISTINCT customer_unique_id)       AS total_customers,
    COUNT(*)                                 AS total_orders,
    MIN(DATE(order_purchase_timestamp))      AS first_order_date,
    MAX(DATE(order_purchase_timestamp))      AS last_order_date,
    ROUND(
        (JULIANDAY(MAX(order_purchase_timestamp)) -
         JULIANDAY(MIN(order_purchase_timestamp))) / 30.44
    , 1)                                     AS observation_months
FROM cof_mat
"""

overview = pd.read_sql(overview_query, conn)
overview

**Note:** `cof_mat` is the materialised version of `customer_order_fact` — it contains one row per delivered order. All 96,478 orders are delivered orders; cancelled and in-progress orders were filtered in the cleaning layer.

---

## 2. Repeat vs One-Time Buyer Split

The core business question: what share of customers ever placed a second order? This uses the **lifetime** definition (no time-window restriction) — a customer is a repeat buyer if they placed at least two delivered orders at any point during the observation period.

In [ ]:
split_query = """
SELECT
    CASE is_repeat_buyer WHEN 1 THEN 'Repeat buyer' ELSE 'One-time buyer' END AS buyer_type,
    COUNT(*)                                 AS customers,
    ROUND(COUNT(*) * 100.0 /
          (SELECT COUNT(*) FROM crsm_mat), 2) AS pct_of_customers,
    ROUND(AVG(days_to_second_order), 1)      AS avg_days_to_second_order
FROM crsm_mat
GROUP BY is_repeat_buyer
ORDER BY is_repeat_buyer DESC
"""

split_df = pd.read_sql(split_query, conn)
split_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# ── Left: Donut chart ─────────────────────────────────────────────────────────
ax_donut = axes[0]
counts = split_df["customers"].values
labels = split_df["buyer_type"].values
colors = [C_BLUE, C_GREY]

wedges, texts, autotexts = ax_donut.pie(
    counts,
    labels=labels,
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.78,
    wedgeprops=dict(width=0.55, edgecolor="white"),
)
for t in autotexts:
    t.set_fontsize(11)
    t.set_fontweight("bold")
ax_donut.set_title("Customer Split by Purchase Behaviour")

# ── Right: Revenue comparison ──────────────────────────────────────────────────
rev_query = """
SELECT buyer_type, avg_revenue_90d, avg_revenue_180d, avg_first_order_value
FROM repeat_vs_one_time_summary
"""
rev_df = pd.read_sql(rev_query, conn)

ax_bar = axes[1]
x = [0, 1]
bar_w = 0.28
cols_to_plot = ["avg_revenue_90d", "avg_revenue_180d"]
bar_colors   = [C_BLUE, C_ORANGE]
bar_labels   = ["Avg Revenue 90d", "Avg Revenue 180d"]

for i, (col, color, label) in enumerate(zip(cols_to_plot, bar_colors, bar_labels)):
    offset = (i - 0.5) * bar_w
    bars = ax_bar.bar(
        [xi + offset for xi in x],
        rev_df[col],
        width=bar_w,
        color=color,
        label=label,
        alpha=0.85,
    )
    for bar in bars:
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 3,
            f"R${bar.get_height():.0f}",
            ha="center", va="bottom", fontsize=8.5,
        )

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(rev_df["buyer_type"].str.replace("_", " ").str.title())
ax_bar.set_ylabel("Average Revenue (R$)")
ax_bar.set_title("Revenue per Customer: Repeat vs One-Time")
ax_bar.legend(fontsize=8.5)
ax_bar.set_ylim(0, rev_df[cols_to_plot].max().max() * 1.2)

plt.tight_layout()
plt.savefig("../images/repeat_vs_onetime_split.png", bbox_inches="tight")
plt.show()
print("Saved: images/repeat_vs_onetime_split.png")

**Interpretation:**  
Only **3.0%** of customers placed a second order — 2,801 out of 93,358. The vast majority (97%) are one-time buyers.  
Despite being a small group, repeat buyers generate meaningfully higher average revenue: **R$254** over 90 days vs **R$161** for one-time buyers — a **58% uplift**. Their 180-day revenue is higher still (R$278), reflecting continued purchasing after the first repeat.  
This confirms that even a modest improvement in repeat rate translates directly into revenue uplift.

---

## 3. Distribution of Orders per Customer

How many customers placed exactly 1, 2, 3… orders? This shows the drop-off cliff after the first purchase — the steeper the cliff, the more acute the retention problem.

In [ ]:
freq_query = """
SELECT
    total_orders_lifetime                    AS orders_placed,
    COUNT(*)                                 AS customers,
    ROUND(COUNT(*) * 100.0 /
          (SELECT COUNT(*) FROM crsm_mat), 2) AS pct_of_customers
FROM crsm_mat
GROUP BY total_orders_lifetime
ORDER BY total_orders_lifetime
"""

freq_df = pd.read_sql(freq_query, conn)
print(freq_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

# Cap display at 7+ orders to keep chart readable
plot_df = freq_df[freq_df["orders_placed"] <= 7].copy()
gt7 = freq_df[freq_df["orders_placed"] > 7]["customers"].sum()
if gt7 > 0:
    plot_df = pd.concat([
        plot_df,
        pd.DataFrame({"orders_placed": ["8+"], "customers": [gt7], "pct_of_customers": [None]})
    ], ignore_index=True)

colors = [C_GREY if str(o) != "1" else C_BLUE for o in plot_df["orders_placed"]]
bars = ax.bar(
    plot_df["orders_placed"].astype(str),
    plot_df["customers"],
    color=colors,
    edgecolor="white",
    width=0.65,
)

# Annotate bars
for bar, (_, row) in zip(bars, plot_df.iterrows()):
    pct = row["pct_of_customers"]
    if pct is not None:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 300,
            f"{pct:.1f}%",
            ha="center", va="bottom", fontsize=8.5,
        )

ax.set_xlabel("Number of Orders Placed (lifetime)")
ax.set_ylabel("Number of Customers")
ax.set_title("Order Frequency Distribution — Drop-off After First Purchase")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Annotation: the retention cliff
ax.annotate(
    "97% of customers\nnever return",
    xy=(0, freq_df.loc[freq_df["orders_placed"] == 1, "customers"].values[0]),
    xytext=(1.5, 70000),
    fontsize=9,
    color=C_BLUE,
    fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C_BLUE, lw=1.2),
)

plt.tight_layout()
plt.savefig("../images/order_frequency_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: images/order_frequency_distribution.png")

**Interpretation:**  
The drop-off after the first purchase is severe. 93,358 customers placed one order; only 2,801 (3%) placed a second. The fall from 2nd to 3rd order is equally steep — only 228 customers reached three orders.  
This is not a funnel that gradually leaks — it's a cliff. The first-to-second conversion is the single biggest opportunity in the business.

---

## 4. Monthly First-Order Trend

How has customer acquisition (first orders) grown over the observation window? This establishes the growth trajectory of the business before overlaying repeat purchase behaviour.

In [ ]:
trend_df = pd.read_sql(
    "SELECT * FROM repeat_purchase_trend_monthly ORDER BY order_month",
    conn
)

# Exclude 2016-09 and 2016-12 (1 order each — incomplete months)
trend_plot = trend_df[
    ~trend_df["order_month"].isin(["2016-09", "2016-12"])
].copy()

fig, ax = plt.subplots(figsize=(12, 4.5))

ax.fill_between(
    trend_plot["order_month"],
    trend_plot["first_orders"],
    alpha=0.15,
    color=C_BLUE,
)
ax.plot(
    trend_plot["order_month"],
    trend_plot["first_orders"],
    color=C_BLUE,
    linewidth=2,
    marker="o",
    markersize=4,
    label="First orders",
)

ax.set_xticks(range(len(trend_plot)))
ax.set_xticklabels(trend_plot["order_month"], rotation=45, ha="right")
ax.set_ylabel("Customers (First Order)")
ax.set_title("Monthly First-Order Volume — New Customer Acquisition")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Mark Nov 2017 peak (Black Friday effect)
nov17_idx = trend_plot[trend_plot["order_month"] == "2017-11"].index
if len(nov17_idx) > 0:
    pos = trend_plot.index.get_loc(nov17_idx[0])
    nov_val = trend_plot.loc[nov17_idx[0], "first_orders"]
    ax.annotate(
        f"Nov 2017 peak\n{nov_val:,} new customers",
        xy=(pos, nov_val),
        xytext=(pos - 3, nov_val - 1500),
        fontsize=8.5,
        color=C_ORANGE,
        arrowprops=dict(arrowstyle="->", color=C_ORANGE, lw=1.2),
    )

ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("../images/monthly_first_order_trend.png", bbox_inches="tight")
plt.show()
print("Saved: images/monthly_first_order_trend.png")

**Interpretation:**  
First-order volume grew steadily from ~700/month in early 2017 to 6,000–7,000/month by 2018. There is a clear spike in **November 2017** (~7,060 new customers) likely driven by Black Friday promotions.  
The business was in a strong acquisition phase — but this makes the low repeat rate more costly, since each wave of newly acquired customers is failing to convert into recurring revenue.

---

## 5. Monthly Second-Order Trend vs First Orders

Overlaying second-order volume against first orders reveals whether repeat purchasing grew in proportion to acquisition — or lagged behind.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

x_labels = trend_plot["order_month"].tolist()
x_pos    = range(len(x_labels))

# Primary axis: first orders (bar)
ax1.bar(
    x_pos,
    trend_plot["first_orders"],
    color=C_BLUE,
    alpha=0.35,
    label="First orders (left axis)",
    width=0.75,
)
ax1.set_ylabel("First Orders", color=C_BLUE)
ax1.tick_params(axis="y", labelcolor=C_BLUE)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Secondary axis: second orders (line)
ax2 = ax1.twinx()
ax2.plot(
    x_pos,
    trend_plot["second_orders"],
    color=C_ORANGE,
    linewidth=2,
    marker="o",
    markersize=4,
    label="Second orders (right axis)",
)
ax2.set_ylabel("Second Orders", color=C_ORANGE)
ax2.tick_params(axis="y", labelcolor=C_ORANGE)
ax2.spines["right"].set_visible(True)
ax2.spines["top"].set_visible(False)

ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=45, ha="right")
ax1.set_title("First Orders vs Second Orders — Monthly Volume")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8.5)

plt.tight_layout()
plt.savefig("../images/monthly_first_vs_second_orders.png", bbox_inches="tight")
plt.show()
print("Saved: images/monthly_first_vs_second_orders.png")

**Interpretation:**  
Second orders did grow in absolute terms (from ~20/month in early 2017 to ~200/month in 2018), but they remained a tiny fraction of first orders throughout. Even in the best months, second orders were roughly 3–4% of first orders placed that month — not as a cohort rate, but in raw volume.  
The gap widened as the business scaled: by 2018, the platform was acquiring 6,000–7,000 new customers per month but converting only 180–240 of them to a second purchase — a structural retention problem, not a seasonal one.

> **Note on 2018 recent months:** The cohort-based `repeat_buyers_identified` column shows low numbers for mid-to-late 2018 cohorts. This is expected — those customers have not had sufficient time to repeat within the observation window (right-censoring effect). The `second_orders` column counts actual second orders placed in that month, which is not affected by censoring.

---

## 6. Repeat Purchase Rate Summary

The headline repeat purchase rates, calculated using right-censored windows. Customers whose first purchase was too recent to have a fair chance at repeating within the window are excluded from the denominator (assigned NULL in the source mart, automatically excluded from AVG).

| Window | Eligible Customers | Repeat Rate |
|--------|-------------------|-------------|
| 30 days | 86,909 | 1.59% |
| 60 days | 81,265 | 1.96% |
| 90 days | 75,387 | 2.28% |
| Lifetime (raw) | 93,358 | 3.00% |

In [ ]:
kpi_query = """
SELECT
    total_valid_orders,
    total_customers,
    repeat_buyers,
    ROUND(raw_repeat_purchase_rate * 100, 2)  AS raw_repeat_rate_pct,
    eligible_customers_30d,
    ROUND(repeat_rate_30d * 100, 2)           AS repeat_rate_30d_pct,
    eligible_customers_60d,
    ROUND(repeat_rate_60d * 100, 2)           AS repeat_rate_60d_pct,
    eligible_customers_90d,
    ROUND(repeat_rate_90d * 100, 2)           AS repeat_rate_90d_pct,
    avg_days_to_second_purchase,
    median_days_to_second_purchase,
    avg_revenue_90d,
    avg_revenue_180d
FROM executive_kpi_summary
"""

kpi_df = pd.read_sql(kpi_query, conn).T
kpi_df.columns = ["Value"]
kpi_df.index.name = "Metric"
kpi_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# ── Left: Window repeat rates bar chart ───────────────────────────────────────
ax_rates = axes[0]
windows  = ["30-day", "60-day", "90-day", "Lifetime\n(raw)"]
rates    = [1.59, 1.96, 2.28, 3.00]
bar_cols = [C_GREY, C_GREY, C_GREY, C_BLUE]

bars = ax_rates.bar(windows, rates, color=bar_cols, width=0.5, edgecolor="white")
for bar, rate in zip(bars, rates):
    ax_rates.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{rate:.2f}%",
        ha="center", va="bottom", fontsize=10, fontweight="bold",
    )

ax_rates.set_ylabel("Repeat Purchase Rate (%)")
ax_rates.set_title("Repeat Purchase Rate by Window")
ax_rates.set_ylim(0, 4.0)

ax_rates.text(
    -0.4, 3.5,
    "Right-censored:\neligible customers only",
    fontsize=7.5, color="#64748B",
    style="italic",
)

# ── Right: Days to second purchase ────────────────────────────────────────────
days_query = """
SELECT days_to_second_order
FROM crsm_mat
WHERE days_to_second_order IS NOT NULL
"""
days_df = pd.read_sql(days_query, conn)

ax_days = axes[1]
ax_days.hist(
    days_df["days_to_second_order"],
    bins=40,
    color=C_BLUE,
    alpha=0.75,
    edgecolor="white",
)
ax_days.axvline(29, color=C_ORANGE, linewidth=1.8, linestyle="--",
                label=f"Median: 29 days")
ax_days.axvline(81.2, color=C_GREEN, linewidth=1.8, linestyle="--",
                label=f"Mean: 81 days")
ax_days.set_xlabel("Days Between First and Second Order")
ax_days.set_ylabel("Number of Customers")
ax_days.set_title("Time to Second Purchase Distribution (Repeat Buyers Only)")
ax_days.legend(fontsize=8.5)
ax_days.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("../images/repeat_rate_and_days_to_second.png", bbox_inches="tight")
plt.show()
print("Saved: images/repeat_rate_and_days_to_second.png")

**Interpretation:**  
The right-censored 90-day repeat rate is **2.28%** — meaning roughly 1 in 44 customers who had a full 90-day observation window placed a second order within it. Expanding to the full observation period (lifetime raw rate) raises this to **3.00%**, still very low.  

The time-to-second-purchase distribution is strongly right-skewed. The **median is 29 days** (many repeat buyers return quickly, likely for top-up purchases or planned replenishment), but the **mean is 81 days** (a long tail of customers who waited months).  

This bimodal pattern suggests two distinct repeat-buyer behaviours:  
- **Quick returners** (within 30 days): likely specific-need buyers or category replenishers  
- **Slow returners** (60–180 days): browsed back eventually, perhaps driven by a promotion or new need  

Both groups are worth retaining, but their re-engagement strategies would differ.

---

## 7. Customer Value Distribution (CLV Bands)

How are customers distributed across observed lifetime value tiers? This gives a sense of how concentrated (or spread) the revenue base is.

In [ ]:
clv_df = pd.read_sql(
    "SELECT clv_band, clv_band_order, customers, avg_revenue_180d, "
    "total_lifetime_observed_revenue FROM clv_distribution_summary "
    "ORDER BY clv_band_order",
    conn
)
clv_df["revenue_share_pct"] = (
    clv_df["total_lifetime_observed_revenue"]
    / clv_df["total_lifetime_observed_revenue"].sum() * 100
).round(1)
print(clv_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

band_colors = ["#CBD5E1", "#93C5FD", "#3B82F6", "#1D4ED8", "#1E3A8A"]
band_labels = clv_df["clv_band"].str.replace("_", " ").str.title()

# Left: customers per band
ax1 = axes[0]
bars = ax1.bar(
    band_labels, clv_df["customers"],
    color=band_colors, edgecolor="white", width=0.6,
)
for bar, row in zip(bars, clv_df.itertuples()):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f"{row.customers:,}",
        ha="center", va="bottom", fontsize=8.5,
    )
ax1.set_ylabel("Customers")
ax1.set_title("Customers by CLV Band")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Right: revenue share per band
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    clv_df["total_lifetime_observed_revenue"],
    labels=band_labels,
    colors=band_colors,
    autopct="%1.1f%%",
    startangle=140,
    pctdistance=0.78,
    wedgeprops=dict(edgecolor="white", linewidth=1.2),
)
for t in autotexts:
    t.set_fontsize(8.5)
ax2.set_title("Revenue Share by CLV Band")

plt.tight_layout()
plt.savefig("../images/clv_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: images/clv_distribution.png")

**Interpretation:**  
Nearly half of all customers (43,246 — 46%) fall in the **low CLV band** (< R$100 lifetime revenue). Another 43% are in the **mid band** (R$100–299).  
Only 2,343 customers (2.5%) are in the **top band** (≥ R$700), yet they account for a disproportionate share of total lifetime revenue.  
This concentration is typical of e-commerce: a small number of high-value customers drive an outsized share of revenue — and most of them are likely repeat buyers.

---

## Summary of Key Findings

| Metric | Value |
|--------|-------|
| Total customers | 93,358 |
| Total valid (delivered) orders | 96,478 |
| Repeat buyers (lifetime) | 2,801 (3.0%) |
| One-time buyers | 90,557 (97.0%) |
| Right-censored repeat rate — 30d | 1.59% |
| Right-censored repeat rate — 90d | 2.28% |
| Median days to second purchase | 29 days |
| Mean days to second purchase | 81 days |
| Avg revenue 90d — repeat buyer | R$253 |
| Avg revenue 90d — one-time buyer | R$161 |
| Revenue uplift from repeat | ~58% |
| Observation period | Sep 2016 – Aug 2018 (23 months) |

### What this means for the project

1. **The problem is real and large.** 97% one-time buyer rate is a significant retention failure at scale.
2. **The revenue gap is meaningful.** Repeat buyers generate 58% more revenue per customer over 90 days.
3. **Fast returners are the primary target.** Median return time is 29 days — a short re-engagement window where email/push outreach within the first month would capture most potential repeat buyers.
4. **Acquisition is healthy but retention doesn't follow.** The business grew aggressively in 2017–2018, but repeat volume grew only marginally. Without retention improvement, CAC efficiency will erode.

---
*Next notebook: `03_retention_and_cohorts.ipynb` — cohort heatmap, monthly retention curves, and time-to-second-purchase by cohort.*

In [ ]:
conn.close()
print("Database connection closed.")